# 04. Hybrid Retrieval & Reciprocal Rank Fusion (RRF)

This notebook demonstrates the hybrid retrieval architecture for the Vachanamrut RAG system (`src/step4_retriever.py`).

### Key Demonstration Steps:
1. **Dense Vector Search**: Performing similarity search over persistent ChromaDB embeddings.
2. **BM25 Keyword Search**: Initializing lexical term-matching over structural document chunks.
3. **Reciprocal Rank Fusion (RRF)**: Merging candidate document lists using reciprocal rank scoring to produce optimal reranked context blocks.
4. **Grounded Synthesis**: Generating a spiritual response using `gpt-4o-mini` with inline scriptural citation constraints.

In [ ]:
import sys
from pathlib import Path

# Add project root to path for modular imports
sys.path.append("..")

from src.step4_retriever import (
    load_chunks_from_json,
    build_bm25_retriever,
    reciprocal_rank_fusion,
    answer_question,
)

## Step 1: Lexical Candidate Pool Generation (BM25)

Load structural document chunks and initialize the `BM25Retriever` to fetch keyword-matched candidates.

In [ ]:
# Load structural chunks for BM25 initialization
chunks = load_chunks_from_json(store_type="structural")
bm25_retriever = build_bm25_retriever(chunks, k=5)

sample_query = "Why am i not able to succeed in my exams?"
bm25_results = bm25_retriever.invoke(sample_query)

print(f"Query: '{sample_query}'\n")
print(f"Top BM25 Candidate Match:")
print(f"Chapter : {bm25_results[0].metadata.get('chapter')}")
print(f"Snippet : {bm25_results[0].page_content[:150]}...\n")


## Step 2: Reciprocal Rank Fusion (RRF) Algorithm Demonstration

Demonstrate how duplicate documents fetched across BM25 (lexical) and Chroma (dense) pools are combined using reciprocal rank weighting:

$$RRF\_Score(d) = \sum_{m \in M} \frac{w_m}{k + r_m(d)}$$

In [ ]:
# Create dummy document lists representing candidate outputs from two retrievers
doc_a = bm25_results[0]
doc_b = bm25_results[1] if len(bm25_results) > 1 else doc_a

dense_pool = [doc_b, doc_a]  # Reversed order
bm25_pool = [doc_a, doc_b]

fused_docs = reciprocal_rank_fusion([bm25_pool, dense_pool], weights=[0.5, 0.5], top_n=2)

print("--- RRF Fusion Results ---")
for idx, doc in enumerate(fused_docs, 1):
    print(f"Rank {idx}: {doc.metadata.get('chapter')}")

## Step 3: Full End-to-End Hybrid RAG Pipeline

Run `answer_question` in `"hybrid"` mode to inspect the complete context retrieval and grounded response generation workflow.

In [ ]:
test_query = "How do i strengthen my faith in God?" 
""

print(f"Query: {test_query}\n")
response = answer_question(test_query, retriever_type="hybrid", store_type="structural", k=4)

print("--- Grounded Answer Output ---")
print(response)